# Orientation: Mahony Filter using Quaternions

The complementary filter employed earlier was insufficient for the project needs. Therefore, here the Mahony filter is tested out to try to remedy its shortcomings.

## Rotations in 3D space: quaternions

Quaternions are an extension of the complex numbers that live in the four-dimensional space. They are useful because they can be used to represent rotations in 3D space, while, at the same time, avoiding the gimbal lock problem associated with Euler angles. They can be represented as:
$$
\bm{q} = (q_0, q_1, q_2, q_3) = q_0 + q_1 \bm{i} + q_2 \bm{j} + q_3 \bm{k} = (s, \bm{v})
$$
where, in the last notation, $s = q_0$ is the "scalar part" and $\bm{v} = ( q_1, q_2, q_3 )$ is the "vector part". 

Quaternions support addition, subtraction, multiplication, and division, similar to complex numbers. Addition and subtraction work on the components individually. Division is not necessary for this application. For multiplication, the associative property as well as the following identities can be used:
$$
\bm{i}^2 = \bm{j}^2 = \bm{k}^2 = -1 \\
\bm{i} \bm{j} = - \bm{j} \bm{i} = \bm{k} \\
\bm{j} \bm{k} = - \bm{k} \bm{j} = \bm{i}, \\
\bm{k} \bm{i} = - \bm{i} \bm{k} = \bm{j} \\
$$
Note inmediately from these identities that the product is not commutative. Let $\bm{p} = (w, \bm{u})$ be another quaternion. A formula in matrix form for the product can be derived:
$$
\bm{q} \otimes \bm{p} =
\begin{bmatrix}
q_0 & -q_1 & -q_2 & -q_3 \\
q_1 & q_0 & -q_3 & q_2 \\
q_2 & q_3 & q_0 & -q_1 \\
q_3 & -q_2 & q_1 & q_0
\end{bmatrix}
\begin{bmatrix}
p_0 \\
p_1 \\
p_2 \\
p_3
\end{bmatrix} =
\begin{bmatrix}
s w - \bm{v} \cdot \bm{u} \\
s \bm{u} + w \bm{v} + \bm{v} \times \bm{u} \\
\end{bmatrix} 
\ne \bm{p} \otimes \bm{q}
$$

The norm of a quaternion $\bm{q}$ is simply the Euclidean norm of its components:
$$
\|\bm{q}\| = \sqrt{q_0^2 + q_1^2 + q_2^2 + q_3^2}
$$
It can be used to normalize a quaternion:
$$
\hat{\bm{q}} = \frac{\bm{q}}{\|\bm{q}\|}
$$

The conjugate of a quaternion $\bm{q}$ is obtained by negating its vector part:
$$
\bm{q}^* = (q_0, -q_1, -q_2, -q_3)
$$

To rotate a vector $\bm{u} \in \mathbb{R}^3$ using a unit quaternion $\hat{\bm{q}}$, first form a pure quaternion $\bm{p} = (0, \bm{u})$, then apply the rotation as follows:
$$
\bm{p}' = (0, \bm{u}') = \hat{\bm{q}} \otimes \bm{p} \otimes \hat{\bm{q}}^*
$$
Here $\hat{\bm{q}}$ can be expressed as:
$$
\hat{\bm{q}} = \cos(\frac{\beta}{2}) + \sin(\frac{\beta}{2}) (n_1 \bm{i} + n_2 \bm{j} + n_3 \bm{k})
$$
Using this notation, $\hat{\bm{n}} = (n_1, n_2, n_3) \in \mathbb{R}^3$ represents the axis of rotation, and rotation is performed by an angle $\beta$ around this axis, or equivalently, on the plane perpendicular to this axis. This can easily be seen interactively in the source [2]. Alternatively, using vector algebra relations [5] and half-angle trigonometric formulas [6], the quaternion that rotates a vector $\bm{u}$ to the direction of a vector $\bm{u'}$ using the shortest arc possible is:
$$
\hat{\bm{q}}
= (\cos(\frac{\beta}{2}), \sin(\frac{\beta}{2}) \hat{\bm{n}})
= \left(\sqrt{\frac{1 + \cos(\beta)}{2}}, \sqrt{\frac{1 - \cos(\beta)}{2}} \frac{\bm{n}}{\|{\bm{n}}\|} \right)
= \left(\sqrt{\frac{1 + \bm{u} \cdot \bm{u'}}{2}}, \sqrt{\frac{1 - \bm{u} \cdot \bm{u'}}{2}} \frac{\bm{u} \times \bm{u'}}{\|{\bm{u} \times \bm{u'}}\|} \right)
$$
Note that this formula assumes that $\bm{u}$ and $\bm{u'}$ are not collinear (i.e. $\bm{u} \times \bm{u'} \neq \bm{0}$). If they are collinear, the rotation axis is not uniquely defined. If $\bm{u} = \bm{u'}$, use $\hat{\bm{n}} = \bm{0}$. If $\bm{u} = -\bm{u'}$, use the basis vector least aligned with $\bm{u}$ as $\hat{\bm{n}}$.

Quaternions are hard visualize on a time series plot directly, but they can be converted to Euler angles with the following formulas [3]:
$$
\phi = atan2(2(q_0 q_1 + q_2 q_3), 1 - 2(q_1^2 + q_2^2)) \\
\theta = asin(2(q_0 q_2 - q_1 q_3)) \\
\psi = atan2(2(q_0 q_3 + q_1 q_2), 1 - 2(q_2^2 + q_3^2))
$$
To go the opposite way, use:
$$
q_0 = \cos\left(\frac{\phi}{2}\right) \cos\left(\frac{\theta}{2}\right) \cos\left(\frac{\psi}{2}\right) + \sin\left(\frac{\phi}{2}\right) \sin\left(\frac{\theta}{2}\right) \sin\left(\frac{\psi}{2}\right) \\
q_1 = \sin\left(\frac{\phi}{2}\right) \cos\left(\frac{\theta}{2}\right) \cos\left(\frac{\psi}{2}\right) - \cos\left(\frac{\phi}{2}\right) \sin\left(\frac{\theta}{2}\right) \sin\left(\frac{\psi}{2}\right) \\
q_2 = \cos\left(\frac{\phi}{2}\right) \sin\left(\frac{\theta}{2}\right) \cos\left(\frac{\psi}{2}\right) + \sin\left(\frac{\phi}{2}\right) \cos\left(\frac{\theta}{2}\right) \sin\left(\frac{\psi}{2}\right) \\
q_3 = \cos\left(\frac{\phi}{2}\right) \cos\left(\frac{\theta}{2}\right) \sin\left(\frac{\psi}{2}\right) - \sin\left(\frac{\phi}{2}\right) \sin\left(\frac{\theta}{2}\right) \cos\left(\frac{\psi}{2}\right)

$$

Sources:
1. [Wikipedia: Quaternion](https://en.wikipedia.org/wiki/Quaternion)
2. [Website: Visualizing Quaternions - Ben Eater & Grant Sanderson (3Blue1Brown)](https://eater.net/quaternions)
3. [Principles of GNSS, Inertial, and Multisensor Integrated Navigation Systems - Paul D. Groves - Chapter 2](../../docs/misc/Principles%20of%20GNSS,%20Inertial,%20and%20Multisensor%20Integrated%20Navigation%20Systems%20-%20Paul%20Groves.pdf)
4. [YouTube: Visualizing the 4d numbers Quaternions](https://www.youtube.com/watch?v=d4EgbgTm0Bg)
5. [Vector Algebra relations: Angles](https://en.wikipedia.org/wiki/Vector_algebra_relations#Angles)
6. [List of trigonometric identities: Half-angle formulas](https://en.wikipedia.org/wiki/List_of_trigonometric_identities#Half-angle_formulas)

## Sensor fusion: Mahony filter

The Mahony filter is a more advanced algorithm that views sensor fusion as a closed loop control system:

* *Error*: angular velocity error derived from accelerometer readings and the true gravity vector.
* *Set point*: constant and equal to zero.
* *Controller*: linear, proportional-integral (PI).
* *Feedback*: instantaneous angular velocity and acceleration readings from the IMU.
* *Physical system*: athlete translating and rotating in space.
* *Output*: corrected angular velocity, ideally approaches zero when perfectly stationary.

In the original paper, orientation was represented using a "Special Orthogonal group (SO(3))", but in the following implementation quaternions will be used.

The steps of a single iteration of the algorithm are as follows:

1. *Estimate current orientation*: making the parallel with the Euler kinematical equations, it can be shown [6] that the quaternion kinematical equation is given by:
   $$
   \dot{\bm{q}}_{w,i}^b = \frac{1}{2} \bm{q}_{w,i-1}^b \otimes (0, \omega_{x,i}, \omega_{y,i}, \omega_{z,i})
   $$
   This can be numerically integrated to yield the current orientation. Using the trapezoidal rule, the integral can be approximated as:
   $$
   \bm{q}_{w,i}^b = \bm{q}_{w,i-1}^b + \frac{\Delta t}{2} (\dot{\bm{q}}_{w,i-1}^b + \dot{\bm{q}}_{w,i}^b)
   $$
   And it also satisfies the property of quaternion norm preservation:
   $$
   \frac{d}{dt} ||\bm{q}_{w,i}^b||^2 = 0
   $$

2. *Obtain gravity vector*: employing the current orientation estimate, transform the gravity vector from the world frame to the body frame: 
   $$
   \bm{\bm{g}}^{b} = \bm{q}_{w,i}^b \otimes \bm{\bm{g}}^{w} \otimes (\bm{q}_{w,i}^b)^*
   $$

3. *Compute the cost*: employ the cross product between the normalized measured specific force and the normalized estimated gravity vector as a measure of the angular velocity error:
   $$
   \bm{e} = 
   k \left(\frac{\bm{\bm{g}}^{b}}{\|\bm{\bm{g}}^{b}\|} \times -\frac{\bm{\bm{f}}^{b}}{\|\bm{\bm{f}}^{b}\|}\right) = 
   -k \left(\frac{\bm{\bm{f}}^{b}}{\|\bm{\bm{f}}^{b}\|} \times \frac{\bm{\bm{g}}^{b}}{\|\bm{\bm{g}}^{b}\|}\right)
   $$
   Notice that, when the device is stationary, the measured acceleration must resemble the gravity vector ($k = 1$). However, when the device is in motion, there's the influence of dynamic acceleration. A simple way to accomodate for this is to scale down the error by a factor $k \in [0,1)$.

3. *Correct with PI feedback*: compute the gyro bias:
   $$
   \dot{\bm{b}_i} = K_I \cdot \bm{e} \\
   \bm{b}_i = \bm{b}_{i-1} + \frac{\Delta t}{2} (\dot{\bm{b}}_{i-1} + \dot{\bm{b}}_{i})
   $$
   Then correct the measured angular velocity with the estimated bias:
   $$
   \bm{\omega}_{i} \leftarrow \bm{\omega}_{i} - \bm{b}_i + K_P \cdot \bm{e}
   $$
   
4. Re-assign the orientation performing the same operations as in step 1. Normalize the quaternion to ensure it remains a valid rotation representation, as it can be subjected to numerical errors.

The initial rotation $\bm{q}_{w,0}^b$ and bias $\bm{b}_0$ must be set before running the filter. For the rotation, an average of initial stationary acceleration measurements can be used to build an gravity vector, $\bm{\bm{g}}^{b}_0 = -(\bar{f^b_x}, \bar{f^b_y}, \bar{f^b_z})$, and derive an quaternion with the formula discussed above considering $\bm{u} = \bm{g}^w = (0, 0, 1)$ and $\bm{u'} = \bm{g}^{b}_0$. For the bias, an average of initial stationary gyroscope measurements can be used to estimate the initial bias, $\bm{b}_0 = (\bar{\omega^b_{x}}, \bar{\omega^b_{y}}, \bar{\omega^b_{z}})$. Processing should not begin until these initial values are properly set.

Regarding the filter parameters, they can be interpreted as follows:
* *Proportional gain* ($K_P$): controls how aggressively the filter corrects tilt error from the accelerometer. Too high and the filter becomes jittery. Too low and the filter responds sluggishly.
* *Integral gain* ($K_I$): controls how quickly the filter estimates or learns the gyro bias. Too high and the bias estimate will chase noise. Too low and the filter will be undercorrect for real gyro bias. 

They must be tuned for optimal performance. Heuristics conventionally used such as the Ziegler-Nichols method [7] can't be used here as there is no controlled physical system to run experiments on, so trial and error tuning paired with simulation is necessary. 

1. [Nonlinear Complementary Filters on the Special Orthogonal Group](../../docs/misc/Nonlinear%20Complementary%20Filters%20on%20the%20Special%20Orthogonal%20Group%20-%20Robert%20Mahony.pdf)
2. [Complementary vs. Mahony vs. EKF: Choosing the Right Attitude Estimator for Your Drone](https://husainlokhandwala.in/2026/08/09/attitude-filter-comparison.html)
3. [IMU Mahony filter explanation](https://medium.com/@k66115704/imu-mahony-filter-explanation-1ae75bf033ab)
4. [Introducción al control de sistemas dinámicos lineales continuos - Teoría de Control - ISI - UTN FRSF](../../docs/misc/9-%20Introducción%20al%20Control%20de%20SDLC.pdf)
5. [Controlador PID - Teoría de Control - ISI - UTN FRSF](../../docs/misc/10-%20Controladores%20P+I+D.pdf)
6. [[IONLAB Lectures] Quaternion Kinematics](https://www.youtube.com/watch?v=CecyVl9iXKM)
7. [Wikipedia: Ziegler-Nichols method](https://en.wikipedia.org/wiki/Ziegler%E2%80%93Nichols_method)